# 01 · Listen to what is possible (no training)
Runs your clips through other enhancement engines and gives you **blind, loudness-matched** files to compare.
Order matters: DeepFilterNet first (no Python dependencies), Resemble Enhance last (it may reinstall PyTorch).
Use headphones, listen at one fixed volume, and rank each clip for: clarity, naturalness (robotic / watery / echo?), and background voices.
Not yet run by me (no Colab access): if a cell fails, copy me the error text. A GPU helps only for Resemble Enhance
(Runtime > Change runtime type > T4 GPU).

In [ ]:
import os, subprocess, json, random, shutil, glob, pathlib
import numpy as np
!pip -q install pyloudnorm soundfile
!nvidia-smi -L || echo "no GPU (fine for DeepFilterNet; Resemble Enhance will be slow)"
for d in ("/content/clips", "/content/work", "/content/blind"):
    os.makedirs(d, exist_ok=True)

## 1. Upload 1-3 clips
Original recordings (mp3 / mp4 / wav). Pick at least one echoey one and one noisy one.
**Optional:** also upload your web-app export of the same clip, named `NAME-enhanced-webapp.wav`, where NAME is the original file name without its extension.

In [ ]:
from google.colab import files
up = files.upload()
for n, b in up.items():
    open(f"/content/clips/{n}", "wb").write(b)
clips = sorted(p for p in glob.glob("/content/clips/*") if "-enhanced-webapp" not in p)
print("originals:", [os.path.basename(c) for c in clips])
# mono 48 kHz and mono 44.1 kHz copies for the engines
for c in clips:
    name = pathlib.Path(c).stem
    w = f"/content/work/{name}"; os.makedirs(w + "/in48", exist_ok=True); os.makedirs(w + "/in44", exist_ok=True)
    subprocess.run(["ffmpeg","-y","-v","error","-i",c,"-ac","1","-ar","48000",f"{w}/in48/{name}.wav"], check=True)
    subprocess.run(["ffmpeg","-y","-v","error","-i",c,"-ac","1","-ar","44100",f"{w}/in44/{name}.wav"], check=True)

## 2. DeepFilterNet (full-band 48 kHz, MIT/Apache) via its precompiled binary

In [ ]:
import requests
try:
    rel = requests.get("https://api.github.com/repos/Rikorose/DeepFilterNet/releases/latest", timeout=30).json()
    cand = [a for a in rel["assets"] if "deep-filter" in a["name"] and "x86_64" in a["name"] and "linux" in a["name"]
            and not a["name"].endswith((".tar.gz", ".sha256", ".zip"))]
    print("release", rel.get("tag_name"), "assets:", [a["name"] for a in cand])
    open("/content/deep-filter", "wb").write(requests.get(cand[0]["browser_download_url"], timeout=600).content)
    os.chmod("/content/deep-filter", 0o755)
    for c in clips:
        name = pathlib.Path(c).stem; w = f"/content/work/{name}"; os.makedirs(w + "/dfn", exist_ok=True)
        r = subprocess.run(["/content/deep-filter", "-o", w + "/dfn", f"{w}/in48/{name}.wav"], capture_output=True, text=True)
        print(name, "->", os.listdir(w + "/dfn"), r.stderr[-300:])
except Exception as e:
    print("DeepFilterNet step failed (skip it, or send me this message):", repr(e))

## 3. Resemble Enhance (denoise, and denoise + bandwidth/quality restoration; 44.1 kHz)
This installs its own PyTorch. If Colab asks you to restart the session afterwards, restart and re-run only the cells from **1. Upload** onward
(the DeepFilterNet outputs are lost on restart, so run section 2 again if you want it in the comparison).
Check the license of Resemble Enhance on its GitHub page before building anything you ship on it. For listening only, it is fine.

In [ ]:
!pip -q install resemble-enhance --upgrade
for c in clips:
    name = pathlib.Path(c).stem; w = f"/content/work/{name}"
    for mode, flag in (("re_denoise", "--denoise_only"), ("re_enhance", "")):
        os.makedirs(f"{w}/{mode}", exist_ok=True)
        cmd = f"resemble_enhance {w}/in44 {w}/{mode} {flag}"
        print(cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print(r.stderr[-400:])

## 4. Blind, loudness-matched comparison files
Every candidate is converted to mono 48 kHz, aligned in length, and matched to -20 LUFS. Files are named with random letters per clip.
The key is kept out of sight until you run the **reveal** cell.

In [ ]:
import soundfile as sf, pyloudnorm as pyln
from scipy.signal import resample_poly
from math import gcd
def load48(path):
    x, sr = sf.read(path, always_2d=True); x = x.mean(1)
    if sr != 48000:
        g = gcd(sr, 48000); x = resample_poly(x, 48000 // g, sr // g)
    return x
key = {}
for c in clips:
    name = pathlib.Path(c).stem; w = f"/content/work/{name}"
    cands = {"original": f"{w}/in48/{name}.wav"}
    wa = f"/content/clips/{name}-enhanced-webapp.wav"
    if os.path.exists(wa): cands["webapp_gtcrn"] = wa
    for label, folder in (("deepfilternet", "dfn"), ("resemble_denoise", "re_denoise"), ("resemble_enhance", "re_enhance")):
        f = glob.glob(f"{w}/{folder}/*.wav")
        if f: cands[label] = f[0]
    audio = {k: load48(v) for k, v in cands.items()}
    n = min(len(a) for a in audio.values())
    labels = list(audio); random.shuffle(labels)
    meter = pyln.Meter(48000)
    key[name] = {}
    for letter, lab in zip("ABCDEFGH", labels):
        a = audio[lab][:n]
        a = pyln.normalize.loudness(a, meter.integrated_loudness(a), -20.0)
        a = a * min(1.0, 0.98 / max(1e-9, np.abs(a).max()))
        sf.write(f"/content/blind/{name}__{letter}.wav", a, 48000, subtype="PCM_16")
        key[name][letter] = lab
json.dump(key, open("/content/blind_key.json", "w"), indent=1)
print({n: sorted(v) for n, v in key.items()}, "-> files ready in /content/blind (letters only, key hidden)")
shutil.make_archive("/content/blind_pack", "zip", "/content/blind")
files.download("/content/blind_pack.zip")

Listen to each clip's letters in the same order, rank them, and write down what you hear (echo, watery, robotic, harsh, dull, background voices).
**Only then** run the next cell.

In [ ]:
def reveal():
    for n, m in json.load(open("/content/blind_key.json")).items():
        print(n, m)
# reveal()   # <- uncomment and run AFTER listening